# Notebook 12: Portfolio Consolidator & Analyst

## What This Notebook Does
1. **Discovers** all portfolio files in a user-defined folder — any number, any supported format
2. **Consolidates** them into a single unified view, merging duplicate tickers across files
3. **Summarises** in the standard format: Ticker · Type · Allocation % · $ Value
4. **Analyses** the consolidated portfolio using LLM + web search

## Supported Input Formats
| Format | How it's read |
|--------|---------------|
| `.json` | Direct JSON load (NB11 output schema) |
| `.xlsx` / `.xls` | pandas `read_excel` |
| `.csv` | pandas `read_csv` |
| `.pdf` | LLM extraction via file contents |

## Key Design Change from Previous Version
- **Before**: Hard-coded list of 5 specific file paths under `user1/`
- **Now**: Pass any folder path → notebook discovers **all** supported files automatically

## Architecture
```
User Folder
    │
    ├── discover_portfolio_files(folder)   ← scans for all supported files
    │
    ├── load_portfolio_file(path)          ← normalises each file to holdings list
    │
    ├── consolidate_holdings(all_holdings) ← merges duplicates, re-normalises to 100%
    │
    ├── build_summary_table()              ← standard output format
    │
    └── analyse_portfolio()                ← LLM + web search analytics
```

## Three-Agent System
| Notebook | Role | Input | Output |
|----------|------|-------|--------|
| **11** | Profile investor, build portfolio | User conversation | `portfolio.json` |
| **12** (This) | Consolidate & analyse | Any folder of files | Unified report + JSON |
| **13** | Investment education chat | `consolidated_portfolio.json` | Conversational advice |

## How to Use This Notebook

### Step 1 — Point to your portfolio folder
In the **Configuration** cell below, set `PORTFOLIO_FOLDER` to the folder
that contains your investment files:
```python
PORTFOLIO_FOLDER = "../data/user1"          # relative path
PORTFOLIO_FOLDER = "/Users/sara/portfolios" # absolute path
```
The notebook will find **every** `.json`, `.xlsx`, `.xls`, `.csv`, and `.pdf`
file inside that folder automatically — no matter how many there are.

### Step 2 — Run All Cells
Use `Kernel > Restart & Run All`. The notebook will:
1. Scan your folder and list every file it finds
2. Load and normalise each file to a standard holdings schema
3. Merge holdings with the same ticker across all files
4. Re-normalise allocations so they sum to 100%
5. Print the consolidated summary table
6. Run LLM + web-search analytics on the result
7. Save `consolidated_portfolio.json` for Notebook 13

### Step 3 — Interactive Prompt (optional)
The last cell lets you type any follow-up question about your portfolio.

### Prerequisites
```
../.env
  LLM_PROVIDER=openai
  LLM_MODEL=gpt-4.1-mini
  OPENAI_API_KEY=your-key
  SERPER_API_KEY=your-key
```

## Installation

In [ ]:
%pip install langchain langchain-openai langchain-community langgraph \
             python-dotenv pydantic pandas openpyxl pypdf \
             google-search-results --quiet

# PDF support: pypdf + Pillow (Pillow ships with most Python environments).
# Both are pure-Python — no system tools or PATH dependencies.
print('Packages installed')


## Imports

In [1]:
import json
import os
import glob
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

from ai_course_utils import load_llm_from_env, display_config

# Load environment
load_dotenv()
load_dotenv('../.env')

print("Imports successful")

NS: what it reads is:  gpt-4.1-mini
Imports successful


## Configuration

**Change `PORTFOLIO_FOLDER` to point at your investment files.**
Everything else is derived automatically.

In [2]:
# ── USER-DEFINED SETTINGS ─────────────────────────────────────────────────────

# Folder that contains your portfolio files.
# Can be relative (to this notebook) or absolute.
# All .json, .xlsx, .xls, .csv, .pdf files inside will be loaded.
PORTFOLIO_FOLDER = "../data/input/user1"   # <-- change this to your folder

# Where to save the consolidated output for Notebook 13
OUTPUT_FILE = "../data/outputs/consolidated_portfolio.json"

# Total portfolio value in USD (used to compute $ values from percentages).
# Set to None if you don't know — $ Value column will be omitted.
TOTAL_PORTFOLIO_VALUE_USD = None   # e.g. 500_000

# ── DERIVED / CONSTANTS ───────────────────────────────────────────────────────
SUPPORTED_EXTENSIONS = {".json", ".xlsx", ".xls", ".csv", ".pdf"}
DIVIDER = "=" * 70

display_config()
print(f"\nPortfolio folder : {os.path.abspath(PORTFOLIO_FOLDER)}")
print(f"Output file      : {OUTPUT_FILE}")
print(f"Total AUM        : {'${:,.0f}'.format(TOTAL_PORTFOLIO_VALUE_USD) if TOTAL_PORTFOLIO_VALUE_USD else 'Not specified'}")

API CONFIGURATION (.env file)
LLM Provider:    openai
LLM Model:       gpt-4.1-mini
Temperature:     0.0

API Keys Status:
  OpenAI               ✓ Set
  Google               ✗ Not set
  Mistral              ✗ Not set
  Anthropic            ✗ Not set
  Serper (Web Search)  ✓ Set

Data Files:
  Provide file paths as function parameters
  Example: load_use_case_config('your_file.xlsx')

Portfolio folder : /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/ai-powered-move-recommendation/data/input/user1
Output file      : ../data/outputs/consolidated_portfolio.json
Total AUM        : Not specified


## Step 1 — Discover Portfolio Files

Scan `PORTFOLIO_FOLDER` and return every supported file.
No hard-coded paths — works with any number of files.

In [3]:
def discover_portfolio_files(folder: str) -> list[str]:
    """
    Recursively find all supported portfolio files inside `folder`.

    Returns a sorted list of absolute file paths.
    Supported: .json, .xlsx, .xls, .csv, .pdf
    """
    folder_path = Path(folder)
    if not folder_path.exists():
        raise FileNotFoundError(
            f"Portfolio folder not found: {folder_path.resolve()}\n"
            "Please update PORTFOLIO_FOLDER in the Configuration cell."
        )

    found = []
    for ext in SUPPORTED_EXTENSIONS:
        # glob non-recursively in the folder (add ** for subdirectories)
        found.extend(folder_path.glob(f"*{ext}"))

    # Sort by name for reproducible ordering
    return sorted(str(p.resolve()) for p in found)


# ── Run discovery ──────────────────────────────────────────────────────────────
portfolio_files = discover_portfolio_files(PORTFOLIO_FOLDER)

print(DIVIDER)
print(f"DISCOVERED {len(portfolio_files)} PORTFOLIO FILE(S) IN: {PORTFOLIO_FOLDER}")
print(DIVIDER)
for i, f in enumerate(portfolio_files, 1):
    size_kb = os.path.getsize(f) // 1024
    print(f"  {i:2d}. {Path(f).name:<40}  ({size_kb} KB)")

if not portfolio_files:
    print("  ⚠  No supported files found.")
    print(f"     Place .json/.xlsx/.csv/.pdf files in: {PORTFOLIO_FOLDER}")
print(DIVIDER)

DISCOVERED 5 PORTFOLIO FILE(S) IN: ../data/input/user1
   1. Portfolio1.pdf                            (101 KB)
   2. Portfolio2.pdf                            (76 KB)
   3. Portfolio3.pdf                            (91 KB)
   4. Portfolio4.pdf                            (78 KB)
   5. Portfolio5.pdf                            (88 KB)


## Step 2 — Load & Normalise Each File

Each file format is parsed differently but every loader returns the same
normalised schema:

```python
{
    "ticker":         str,   # e.g. "VTI"
    "company_name":   str,   # e.g. "Vanguard Total Stock Market ETF"
    "investment_type": str,  # e.g. "ETF", "Stock", "Bond"
    "allocation_pct": float, # percentage 0-100
    "value_usd":      float | None,
    "source_file":    str,   # which file this came from
}
```

In [4]:
import math, base64, io
from pathlib import Path as _Path

# ── Field aliases (CSV / Excel column name matching) ─────────────────────────
TICKER_ALIASES = {"ticker","symbol","stock","etf","security","asset"}
NAME_ALIASES   = {"company_name","name","security_name","description","holding"}
TYPE_ALIASES   = {"investment_type","type","asset_type","instrument"}
ALLOC_ALIASES  = {"allocation_pct","allocation","percentage","weight","pct",
                   "percent","% of portfolio","percentage of portfolio",
                   "portfolio_pct","alloc_%"}
VALUE_ALIASES  = {"value_usd","value","market_value","$ value","dollar_value",
                   "amount","mkt_value","current_value","amount_usd",
                   "market value","current value"}


def _match_col(columns, aliases):
    col_lower = {c.lower().strip(): c for c in columns}
    for alias in aliases:
        if alias in col_lower:
            return col_lower[alias]
    return None


def _is_missing(v):
    """Safe None / empty / NaN check (IEEE-754 aware)."""
    if v is None or v == "": return True
    try: return math.isnan(float(v))
    except (TypeError, ValueError): return False


def _normalise_row(ticker, name, inv_type, alloc_pct, value_usd,
                   gain_loss_usd, gain_loss_pct, source):
    return {
        "ticker":          str(ticker).upper().strip(),
        "company_name":    str(name).strip() if name else str(ticker).upper().strip(),
        "investment_type": str(inv_type).strip() if inv_type else "Unknown",
        "allocation_pct":  float(alloc_pct)    if not _is_missing(alloc_pct)    else 0.0,
        "value_usd":       float(value_usd)    if not _is_missing(value_usd)    else None,
        "gain_loss_usd":   float(gain_loss_usd) if not _is_missing(gain_loss_usd) else None,
        "gain_loss_pct":   float(gain_loss_pct) if not _is_missing(gain_loss_pct) else None,
        "source_file":     source,
    }


# ── PDF image extraction (pypdf + Pillow — no system tools required) ─────────

def _pdf_to_images(path: str) -> list[str]:
    """
    Extract embedded PNG/JPEG images from a PDF using pypdf.

    Brokerage statements are typically rendered as images embedded in the PDF
    (one image per page). pypdf.page.images accesses these directly from the
    PDF's XObject stream — no rendering, no system PATH, no poppler needed.

    Tested on Portfolio1-5.pdf: each yields 1 image, ~100-135 KB.
    Returns base64-encoded PNG strings.
    """
    from pypdf import PdfReader
    from PIL import Image as _Image

    images = []
    try:
        reader = PdfReader(path)
        for page in reader.pages:
            for img_obj in page.images:
                pil = _Image.open(io.BytesIO(img_obj.data))
                buf = io.BytesIO()
                pil.save(buf, format="PNG")
                images.append(base64.b64encode(buf.getvalue()).decode())
    except Exception as e:
        print(f"  ⚠  Image extraction failed for {_Path(path).name}: {e}")
        return []
    return images


# ── Format-specific loaders ───────────────────────────────────────────────────

def _load_json(path: str) -> list[dict]:
    with open(path) as f:
        data = json.load(f)
    holdings_raw = data.get("holdings", data if isinstance(data, list) else [])
    return [_normalise_row(
        ticker       = h.get("ticker","???"),
        name         = h.get("company_name", h.get("name","")),
        inv_type     = h.get("investment_type", h.get("type","Unknown")),
        alloc_pct    = h.get("allocation_pct", h.get("allocation", h.get("weight",0))),
        value_usd    = h.get("value_usd", h.get("amount_usd", h.get("value",None))),
        gain_loss_usd= h.get("gain_loss_usd", h.get("gain_loss",None)),
        gain_loss_pct= h.get("gain_loss_pct",None),
        source       = Path(path).name,
    ) for h in holdings_raw]


def _load_tabular(path: str) -> list[dict]:
    ext = Path(path).suffix.lower()
    df  = pd.read_excel(path) if ext in (".xlsx",".xls") else pd.read_csv(path)
    df.columns = [str(c) for c in df.columns]
    df = df.dropna(how="all")
    col_ticker = _match_col(df.columns, TICKER_ALIASES)
    if not col_ticker:
        print(f"  ⚠  No Ticker column in {Path(path).name} — skipping.")
        return []
    col_name   = _match_col(df.columns, NAME_ALIASES)
    col_type   = _match_col(df.columns, TYPE_ALIASES)
    col_alloc  = _match_col(df.columns, ALLOC_ALIASES)
    col_value  = _match_col(df.columns, VALUE_ALIASES)
    results = []
    for _, row in df.iterrows():
        ticker = str(row[col_ticker]).strip()
        if not ticker or ticker.lower() in ("nan","none",""): continue
        results.append(_normalise_row(
            ticker        = ticker,
            name          = row[col_name]  if col_name  else None,
            inv_type      = row[col_type]  if col_type  else "Unknown",
            alloc_pct     = row[col_alloc] if col_alloc else 0.0,
            value_usd     = row[col_value] if col_value else None,
            gain_loss_usd = None,
            gain_loss_pct = None,
            source        = Path(path).name,
        ))
    return results


def _load_pdf_via_llm(path: str, llm) -> list[dict]:
    """
    Extract holdings from a PDF using LLM vision.

    Method: pypdf extracts the embedded portfolio screenshot image;
    the image is sent as base64 PNG to a vision-capable LLM which reads
    the table and returns structured JSON — exactly as a human would.

    This approach is robust to image-based PDFs where text extraction
    returns empty strings (the case with all brokerage statements tested).
    """
    fname = Path(path).name
    print(f"    Extracting images from {fname}...")

    images = _pdf_to_images(str(path))
    if not images:
        print(f"  ⚠  No images found in {fname}. Cannot extract holdings.")
        return []

    print(f"    {len(images)} image(s) extracted — sending to LLM vision...")

    # Build multimodal message: all page images + extraction prompt
    content = [
        {"type":"image_url","image_url":{"url":f"data:image/png;base64,{b}"}}
        for b in images
    ]
    content.append({"type":"text","text": """\
You are a financial data extractor reading a brokerage portfolio statement image.

Extract ALL individual investment holdings from the table shown.
Do NOT include Cash, Money Market, or account-level summary rows.

Return a JSON array where each object has exactly these fields:
  "ticker"          - stock/ETF symbol (e.g. "VTI", "BND") — uppercase
  "company_name"    - full security name as shown
  "investment_type" - one of: ETF, Stock, Bond ETF, Mutual Fund, REIT, Other
  "amount_usd"      - dollar Value of holding as plain float (e.g. 46679.0)
  "allocation_pct"  - % of Portfolio if shown, else 0
  "gain_loss_usd"   - Gain/Loss $ as plain float, negative if a loss (e.g. -920.12)
  "gain_loss_pct"   - Gain/Loss % as plain float, negative if a loss (e.g. -2.49)

Rules:
- Return ONLY the raw JSON array — no markdown fences, no explanation.
- All numbers must be plain floats, never formatted strings like "$46,679".
- Ticker symbols must be uppercase."""
    })

    from langchain_core.messages import HumanMessage as _HM
    try:
        response = llm.invoke([_HM(content=content)])
    except Exception as e:
        print(f"  ⚠  LLM vision call failed for {fname}: {e}")
        print(f"     Ensure model supports vision: gpt-4o, gpt-4.1, gpt-4.1-mini")
        return []

    raw = response.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    raw = raw.strip()

    try:
        holdings_raw = json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"  ⚠  Invalid JSON from LLM for {fname}: {e}")
        print(f"     Response preview: {raw[:300]}")
        return []

    if not isinstance(holdings_raw, list):
        print(f"  ⚠  Expected JSON array for {fname}")
        return []

    results = [_normalise_row(
        ticker        = h.get("ticker","???"),
        name          = h.get("company_name",""),
        inv_type      = h.get("investment_type","Unknown"),
        alloc_pct     = h.get("allocation_pct",0),
        value_usd     = h.get("amount_usd", h.get("value_usd",None)),
        gain_loss_usd = h.get("gain_loss_usd",None),
        gain_loss_pct = h.get("gain_loss_pct",None),
        source        = Path(path).name,
    ) for h in holdings_raw]

    print(f"    ✓ {len(results)} holdings extracted from {fname}")
    return results


def load_portfolio_file(path: str, llm=None) -> list[dict]:
    ext = Path(path).suffix.lower()
    if ext == ".json":           return _load_json(path)
    elif ext in (".xlsx",".xls",".csv"): return _load_tabular(path)
    elif ext == ".pdf":
        if llm is None:
            print(f"  ⚠  LLM required to parse PDFs — skipping {Path(path).name}")
            return []
        return _load_pdf_via_llm(path, llm)
    else:
        print(f"  ⚠  Unsupported: {ext} — skipping {Path(path).name}")
        return []


print("Portfolio loaders ready  (PDF: pypdf image → LLM vision)")
print(f"  Supported formats: {', '.join(sorted(SUPPORTED_EXTENSIONS))}")



Portfolio loaders ready  (PDF: pypdf image → LLM vision)
  Supported formats: .csv, .json, .pdf, .xls, .xlsx


## Step 3 — Load All Files

Iterate over every discovered file and collect all holdings.

In [5]:
# Initialise the LLM (needed for PDF extraction)
llm = load_llm_from_env()

all_raw_holdings = []
load_errors      = []

print(DIVIDER)
print(f"LOADING {len(portfolio_files)} FILE(S)")
print(DIVIDER)

for path in portfolio_files:
    fname = Path(path).name
    try:
        holdings = load_portfolio_file(path, llm=llm)
        if holdings:
            all_raw_holdings.extend(holdings)
            print(f"  ✓  {fname:<45} → {len(holdings)} holding(s) loaded")
        else:
            print(f"  ⚠  {fname:<45} → 0 holdings (check file format)")
    except Exception as e:
        load_errors.append((fname, str(e)))
        print(f"  ✗  {fname:<45} → ERROR: {e}")

print(DIVIDER)
print(f"Total raw holdings collected : {len(all_raw_holdings)}")
if load_errors:
    print(f"Files with errors            : {len(load_errors)}")
print(DIVIDER)

🤖 Loading LLM: openai / gpt-4.1-mini
LOADING 5 FILE(S)
    Extracting images from Portfolio1.pdf...
    1 image(s) extracted — sending to LLM vision...
    ✓ 6 holdings extracted from Portfolio1.pdf
  ✓  Portfolio1.pdf                                → 6 holding(s) loaded
    Extracting images from Portfolio2.pdf...
    1 image(s) extracted — sending to LLM vision...
    ✓ 5 holdings extracted from Portfolio2.pdf
  ✓  Portfolio2.pdf                                → 5 holding(s) loaded
    Extracting images from Portfolio3.pdf...
    1 image(s) extracted — sending to LLM vision...
    ✓ 5 holdings extracted from Portfolio3.pdf
  ✓  Portfolio3.pdf                                → 5 holding(s) loaded
    Extracting images from Portfolio4.pdf...
    1 image(s) extracted — sending to LLM vision...
    ✓ 6 holdings extracted from Portfolio4.pdf
  ✓  Portfolio4.pdf                                → 6 holding(s) loaded
    Extracting images from Portfolio5.pdf...
    1 image(s) extracted — sendi

## Step 4 — Consolidate Holdings

Merge duplicate tickers across all files:
- Sum `allocation_pct` across files
- Sum `value_usd` where available
- Re-normalise allocations to exactly 100%
- Use the most descriptive `company_name` seen for each ticker

In [6]:
def consolidate_holdings(raw_holdings: list[dict]) -> list[dict]:
    """
    Merge holdings with the same ticker across all source files.

    Allocation strategy (priority order):
      1. Dollar values (value_usd) present → compute allocation_pct from
         holding_usd / total_usd * 100. Most accurate — matches brokerage math.
      2. Only allocation_pct available → sum and re-normalise to 100%.
      3. Neither → equal weight fallback.

    Gain/loss: summed in dollar terms; percentage averaged across appearances.
    """
    merged: dict[str, dict] = {}

    for h in raw_holdings:
        ticker = h["ticker"]
        if ticker == "???" or not ticker:
            continue
        if ticker not in merged:
            merged[ticker] = {
                "ticker":          ticker,
                "company_name":    h["company_name"],
                "investment_type": h["investment_type"],
                "allocation_pct":  0.0,
                "value_usd":       0.0,
                "gain_loss_usd":   0.0,
                "gl_pct_list":     [],
                "has_value":       False,
                "source_files":    [],
            }
        e = merged[ticker]
        if len(h["company_name"]) > len(e["company_name"]):
            e["company_name"] = h["company_name"]
        if e["investment_type"] == "Unknown" and h["investment_type"] != "Unknown":
            e["investment_type"] = h["investment_type"]
        if not _is_missing(h.get("allocation_pct")):
            e["allocation_pct"] += float(h["allocation_pct"])
        if not _is_missing(h.get("value_usd")):
            e["value_usd"] += float(h["value_usd"])
            e["has_value"] = True
        if not _is_missing(h.get("gain_loss_usd")):
            e["gain_loss_usd"] += float(h["gain_loss_usd"])
        if not _is_missing(h.get("gain_loss_pct")):
            e["gl_pct_list"].append(float(h["gain_loss_pct"]))
        if h["source_file"] not in e["source_files"]:
            e["source_files"].append(h["source_file"])

    holdings_list = list(merged.values())

    # ── Priority 1: allocation from dollar values ──────────────────────────
    has_any_value = any(h["has_value"] for h in holdings_list)
    if has_any_value:
        total_usd = sum(h["value_usd"] for h in holdings_list)
        if total_usd > 0:
            for h in holdings_list:
                h["allocation_pct"] = round(h["value_usd"] / total_usd * 100, 4)
            # Fix floating-point drift
            diff = round(100.0 - sum(h["allocation_pct"] for h in holdings_list), 4)
            if diff:
                top = max(holdings_list, key=lambda x: x["allocation_pct"])
                top["allocation_pct"] = round(top["allocation_pct"] + diff, 4)

    # ── Priority 2: re-normalise raw allocation_pct ────────────────────────
    else:
        total_alloc = sum(h["allocation_pct"] for h in holdings_list)
        if total_alloc > 0:
            for h in holdings_list:
                h["allocation_pct"] = round(h["allocation_pct"] / total_alloc * 100, 4)

    # ── Priority 3: TOTAL_PORTFOLIO_VALUE_USD fallback ─────────────────────
    if TOTAL_PORTFOLIO_VALUE_USD and not has_any_value:
        for h in holdings_list:
            h["value_usd"]  = round(TOTAL_PORTFOLIO_VALUE_USD * h["allocation_pct"] / 100, 2)
            h["has_value"]  = True

    # ── Finalise gain/loss, clean up ─────────────────────────────────────────
    for h in holdings_list:
        if not h["has_value"]:
            h["value_usd"] = None
        del h["has_value"]
        h["gain_loss_pct"] = (round(sum(h["gl_pct_list"])/len(h["gl_pct_list"]),2)
                              if h["gl_pct_list"] else None)
        del h["gl_pct_list"]

    holdings_list.sort(key=lambda x: x["allocation_pct"], reverse=True)
    return holdings_list


consolidated = consolidate_holdings(all_raw_holdings)
total_aum  = sum(h["value_usd"]    for h in consolidated if h["value_usd"])
total_gain = sum(h["gain_loss_usd"] for h in consolidated if h["gain_loss_usd"])

print(DIVIDER)
print("CONSOLIDATION COMPLETE")
print(DIVIDER)
print(f"  Raw holdings (before merge)  : {len(all_raw_holdings)}")
print(f"  Unique tickers (after merge) : {len(consolidated)}")
print(f"  Total AUM                    : ${total_aum:,.0f}")
print(f"  Total Gain/Loss              : +${total_gain:,.0f}")
multi = [h for h in consolidated if len(h["source_files"]) > 1]
if multi:
    print(f"  Tickers merged across files  : {len(multi)}")
    for h in multi:
        print(f"    {h['ticker']:<8} in {h['source_files']}")
print(DIVIDER)



CONSOLIDATION COMPLETE
  Raw holdings (before merge)  : 27
  Unique tickers (after merge) : 17
  Total AUM                    : $979,963
  Total Gain/Loss              : +$135,237
  Tickers merged across files  : 7
    VTI      in ['Portfolio1.pdf', 'Portfolio3.pdf', 'Portfolio5.pdf']
    NVDA     in ['Portfolio2.pdf', 'Portfolio4.pdf']
    BND      in ['Portfolio1.pdf', 'Portfolio3.pdf']
    MSFT     in ['Portfolio2.pdf', 'Portfolio4.pdf']
    VEA      in ['Portfolio1.pdf', 'Portfolio3.pdf', 'Portfolio5.pdf']
    VWO      in ['Portfolio1.pdf', 'Portfolio4.pdf', 'Portfolio5.pdf']
    SPY      in ['Portfolio2.pdf', 'Portfolio4.pdf']


## Step 5 — Consolidated Portfolio Summary

Standard output format:

| Ticker | Type | % Allocation | $ Value |
|--------|------|-------------|---------|

In [7]:
def print_consolidated_summary(holdings: list[dict]) -> None:
    """Print the standard consolidated portfolio summary.

    Uses only json and Python builtins — no third-party table library.
    """
    if not holdings:
        print("No holdings to display.")
        return

    has_value = any(h["value_usd"] is not None for h in holdings)

    # ── Column widths (derived from data so the table always fits) ────────
    col_ticker = max(max(len(h["ticker"]) for h in holdings), 6)
    col_name   = max(min(max(len(h["company_name"]) for h in holdings), 36), 4)
    col_type   = max(max(len(h["investment_type"]) for h in holdings), 4)

    def fmt_name(name: str) -> str:
        return (name[:33] + "...") if len(name) > 36 else name

    # ── Header ─────────────────────────────────────────────────────────────
    header = (
        f"  {'Ticker':<{col_ticker}}  {'Name':<{col_name}}  "
        f"{'Type':<{col_type}}  {'Alloc %':>8}"
    )
    if has_value:
        header += f"  {'$ Value':>14}"
    sep = "-" * len(header)

    print(DIVIDER)
    print("CONSOLIDATED PORTFOLIO SUMMARY")
    print(DIVIDER)
    print(header)
    print(sep)

    # ── Rows ───────────────────────────────────────────────────────────────
    for h in holdings:
        row = (
            f"  {h['ticker']:<{col_ticker}}  "
            f"{fmt_name(h['company_name']):<{col_name}}  "
            f"{h['investment_type']:<{col_type}}  "
            f"{h['allocation_pct']:>7.2f}%"
        )
        if has_value:
            val_str = f"${h['value_usd']:>13,.0f}" if h["value_usd"] else f"{'N/A':>14}"
            row += f"  {val_str}"
        print(row)

    # ── Totals ─────────────────────────────────────────────────────────────
    print(sep)
    total_pct = sum(h["allocation_pct"] for h in holdings)
    totals = f"  {'TOTAL':<{col_ticker}}  {len(holdings)} holdings  {total_pct:>8.2f}%"
    if has_value:
        total_val = sum(h["value_usd"] for h in holdings if h["value_usd"])
        totals += f"  ${total_val:>13,.0f}"
    print(totals)
    print(DIVIDER)

    # ── Asset-type breakdown ───────────────────────────────────────────────
    type_totals: dict[str, float] = {}
    for h in holdings:
        t = h["investment_type"]
        type_totals[t] = type_totals.get(t, 0) + h["allocation_pct"]
    print("ALLOCATION BY ASSET TYPE")
    for t, pct in sorted(type_totals.items(), key=lambda x: -x[1]):
        bar = "\u2588" * int(pct / 2)
        print(f"  {t:<15}  {pct:6.2f}%  {bar}")
    print(DIVIDER)

    # ── Full JSON output ───────────────────────────────────────────────────
    print("CONSOLIDATED PORTFOLIO JSON")
    print(DIVIDER)
    print(json.dumps(holdings, indent=2))
    print(DIVIDER)


print_consolidated_summary(consolidated)

CONSOLIDATED PORTFOLIO SUMMARY
  Ticker  Name                                  Type       Alloc %         $ Value
----------------------------------------------------------------------------------
  VTI     Vanguard Total Stock Market ETF       ETF         20.85%  $      204,294
  NVDA    NVIDIA                                Stock       12.48%  $      122,321
  BND     Vanguard Total Bond Market ETF        Bond ETF    11.13%  $      109,095
  MSFT    Microsoft                             Stock       10.94%  $      107,207
  AAPL    Apple                                 Stock        8.57%  $       84,022
  IJR     iShares Core S&P Small-Cap ETF        ETF          6.20%  $       60,742
  VEA     Vanguard FTSE Developed Markets ETF   ETF          5.89%  $       57,752
  VTTHX   Vanguard Target Retirement 2035       Other        4.05%  $       39,681
  LQD     iShares iBoxx Investment Grade Co...  Bond ETF     3.68%  $       36,049
  TSLA    Tesla                                 Stock   

## Step 6 — Save Consolidated Portfolio

Save to `consolidated_portfolio.json` for use by Notebook 13 (Education Agent).

In [8]:
def save_consolidated_portfolio(holdings: list[dict], output_path: str) -> None:
    """Save consolidated holdings + metadata to JSON."""
    total_pct = sum(h["allocation_pct"] for h in holdings)
    total_val = sum(h["value_usd"] for h in holdings if h["value_usd"])

    output = {
        "source_folder":     PORTFOLIO_FOLDER,
        "source_files":      [Path(f).name for f in portfolio_files],
        "total_holdings":    len(holdings),
        "total_allocation":  round(total_pct, 4),
        "total_value_usd":   round(total_val, 2) if total_val else None,
        "holdings":          holdings,
    }

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)

    print(f"Consolidated portfolio saved → {output_path}")
    print(f"  Holdings : {len(holdings)}")
    print(f"  Total %  : {total_pct:.2f}%")
    if total_val:
        print(f"  Total $  : ${total_val:,.0f}")


save_consolidated_portfolio(consolidated, OUTPUT_FILE)

Consolidated portfolio saved → ../data/outputs/consolidated_portfolio.json
  Holdings : 17
  Total %  : 100.00%
  Total $  : $979,963


## Step 7 — Portfolio Analytics (LLM + Web Search)

The analytics agent can:
- Search for current prices, performance, and news on any holding
- Assess diversification, concentration risk, and sector balance
- Flag rebalancing opportunities
- Compare the portfolio to a benchmark (e.g. S&P 500)

The consolidation prompt is sent first to get an immediate LLM analysis,
followed by a tool-calling agent for deeper web-augmented insights.

In [9]:
@tool
def search_web(query: str) -> str:
    """Search the web for current market data, news, or fund information."""
    try:
        return GoogleSerperAPIWrapper().run(query)
    except Exception as e:
        return f"Search unavailable: {e}"


@tool
def analyze_portfolio(question_type: str = "consolidated_view") -> str:
    """
    Answer portfolio analysis questions from the consolidated holdings.

    question_type options:
      consolidated_view  — full table: ticker, type, allocation%, value, gain/loss
      biggest_gainers    — top 5 gainers by % and by $ amount with summary
      biggest_losers     — all holdings at a loss, ranked by loss %
      overweight         — asset-class balance vs targets + individual position flags
      summary            — AUM, return, top/bottom performers in one snapshot
    """
    h = consolidated
    if not h:
        return "No portfolio loaded. Run the consolidation cells first."

    total_aum  = sum(x["value_usd"]     for x in h if x["value_usd"])
    total_gain = sum(x["gain_loss_usd"] for x in h if x["gain_loss_usd"])
    total_cost = total_aum - total_gain
    total_ret  = round(total_gain / total_cost * 100, 2) if total_cost > 0 else 0
    q = question_type.lower().strip()

    # ── Consolidated view ─────────────────────────────────────────────────────
    if q in ("consolidated_view", "consolidated", "view", "all"):
        rows = [{"ticker":x["ticker"], "type":x["investment_type"],
                 "allocation_pct":round(x["allocation_pct"],2),
                 "value_usd":round(x["value_usd"],0) if x["value_usd"] else None,
                 "gain_loss_usd":round(x["gain_loss_usd"],0) if x["gain_loss_usd"] else None,
                 "gain_loss_pct":x["gain_loss_pct"],
                 "sources":x["source_files"]}
                for x in h]
        type_totals = {}
        for x in h:
            type_totals.setdefault(x["investment_type"],0)
            type_totals[x["investment_type"]] += x["allocation_pct"]
        return json.dumps({
            "total_holdings":  len(h),
            "total_aum":       round(total_aum,0),
            "total_gain_usd":  round(total_gain,0),
            "total_return_pct":total_ret,
            "holdings":        rows,
            "asset_type_breakdown": {k:round(v,2) for k,v in
                                     sorted(type_totals.items(),key=lambda x:-x[1])},
        }, indent=2)

    # ── Biggest gainers ───────────────────────────────────────────────────────
    elif q in ("biggest_gainers","gainers","top_gainers","gained_most"):
        gainers = sorted([x for x in h if x.get("gain_loss_pct") is not None],
                         key=lambda x: -x["gain_loss_pct"])
        top_dollar = max(h, key=lambda x: x.get("gain_loss_usd") or 0)
        return json.dumps({
            "top_5_by_pct": [{"ticker":x["ticker"],"gain_pct":x["gain_loss_pct"],
                               "gain_usd":round(x["gain_loss_usd"],0),
                               "value_usd":round(x["value_usd"],0),
                               "name":x["company_name"]} for x in gainers[:5]],
            "top_gainer_by_pct": gainers[0]["ticker"] if gainers else None,
            "top_gainer_by_usd": top_dollar["ticker"],
            "top_gain_pct":  gainers[0]["gain_loss_pct"] if gainers else 0,
            "top_gain_usd":  round(top_dollar["gain_loss_usd"],0),
            "portfolio_total_gain_usd": round(total_gain,0),
            "portfolio_total_return_pct": total_ret,
        }, indent=2)

    # ── Biggest losers ────────────────────────────────────────────────────────
    elif q in ("biggest_losers","losers","lost_most","losses"):
        losers = sorted([x for x in h if (x.get("gain_loss_usd") or 0) < 0],
                        key=lambda x: x.get("gain_loss_pct") or 0)
        total_loss = sum(x["gain_loss_usd"] for x in losers)
        return json.dumps({
            "losers": [{"ticker":x["ticker"],"loss_pct":x["gain_loss_pct"],
                        "loss_usd":round(x["gain_loss_usd"],0),
                        "value_usd":round(x["value_usd"],0),
                        "name":x["company_name"]} for x in losers],
            "total_holdings_at_loss": len(losers),
            "total_holdings": len(h),
            "total_loss_usd": round(total_loss,0),
            "biggest_loser_by_pct": losers[0]["ticker"] if losers else None,
            "biggest_loser_by_usd": min(losers,key=lambda x:x["gain_loss_usd"])["ticker"] if losers else None,
            "portfolio_net_gain_usd": round(total_gain,0),
            "portfolio_net_return_pct": total_ret,
        }, indent=2)

    # ── Overweight / underweight ──────────────────────────────────────────────
    elif q in ("overweight","underweight","rebalancing","balance","weight"):
        TARGETS = {"ETF":45.0,"Stock":25.0,"Bond ETF":20.0,"Mutual Fund":5.0,"Other":5.0}
        actual = {}
        for x in h:
            t = x["investment_type"]
            actual.setdefault(t,0)
            actual[t] += x["allocation_pct"]
        type_analysis = {}
        for t,tgt in TARGETS.items():
            act  = round(actual.get(t,0),2)
            diff = round(act - tgt, 2)
            type_analysis[t] = {
                "actual_pct": act, "target_pct": tgt, "diff_pct": diff,
                "status": ("OVERWEIGHT" if diff>5 else "UNDERWEIGHT" if diff<-5
                           else "slightly_over" if diff>2 else "slightly_under" if diff<-2
                           else "on_target"),
            }
        flagged = []
        for x in h:
            a = x["allocation_pct"]
            if a > 15:    flag = "SIGNIFICANTLY_OVERWEIGHT"
            elif a > 10:  flag = "OVERWEIGHT"
            elif a < 1.0: flag = "VERY_SMALL_POSITION"
            else: continue
            flagged.append({"ticker":x["ticker"],"allocation_pct":round(a,2),
                             "value_usd":round(x["value_usd"],0),"flag":flag})
        return json.dumps({
            "asset_class_analysis": type_analysis,
            "flagged_positions":    flagged,
            "tech_concentration_pct": round(sum(x["allocation_pct"] for x in h
                                         if x["ticker"] in ("NVDA","MSFT","AAPL","GOOG")),2),
        }, indent=2)

    # ── Summary ───────────────────────────────────────────────────────────────
    else:
        top3 = sorted([x for x in h if x.get("gain_loss_pct") is not None],
                      key=lambda x: -x["gain_loss_pct"])[:3]
        bot3 = sorted([x for x in h if (x.get("gain_loss_usd") or 0) < 0],
                      key=lambda x: x.get("gain_loss_pct") or 0)[:3]
        return json.dumps({
            "total_aum":        round(total_aum,0),
            "total_gain_usd":   round(total_gain,0),
            "total_return_pct": total_ret,
            "total_holdings":   len(h),
            "top_3_gainers":    [{"ticker":x["ticker"],"gain_pct":x["gain_loss_pct"]} for x in top3],
            "top_3_losers":     [{"ticker":x["ticker"],"loss_pct":x["gain_loss_pct"]} for x in bot3],
        }, indent=2)


# ── Assemble tool list (used by analytics-graph cell) ────────────────────────
tools = [search_web, analyze_portfolio]

print(f"Tools defined: {[t.name for t in tools]}")
print("  analyze_portfolio modes: consolidated_view | biggest_gainers | biggest_losers | overweight | summary")




Tools defined: ['search_web', 'analyze_portfolio']
  analyze_portfolio modes: consolidated_view | biggest_gainers | biggest_losers | overweight | summary


In [11]:
# Build the analytics agent (tool-calling loop — same pattern as NB11)
llm_with_tools = llm.bind_tools(tools)
memory          = MemorySaver()


ANALYTICS_SYSTEM_PROMPT = """\
You are an expert portfolio analyst. You have access to a web search tool
to find current prices, performance data, and market news.

When analysing a portfolio:
1. Review the holdings, allocations, and asset types provided.
2. Use web search to look up recent performance or news on key holdings
   when relevant to the user's question.
3. Provide actionable insights covering:
   - Diversification assessment (sector, geography, asset type)
   - Concentration risk (any single holding >20%?)
   - Rebalancing signals (over/underweight areas)
   - Performance commentary on major holdings
   - Benchmark comparison where possible
4. Always include a brief summary at the end.
5. Remind users this is educational, not financial advice.

Format your responses clearly with section headers.
"""


def analytics_assistant(state: MessagesState):
    """Analytics LLM node — called with portfolio context in system prompt."""
    messages = [SystemMessage(content=ANALYTICS_SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(messages)]}


def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    return "tools" if state["messages"][-1].tool_calls else "__end__"


# Build graph
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("assistant", analytics_assistant)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "assistant")
graph_builder.add_conditional_edges("assistant", should_continue)
graph_builder.add_edge("tools", "assistant")

analytics_graph = graph_builder.compile(checkpointer=memory)
print("Analytics agent graph compiled")


def build_portfolio_context(holdings: list[dict]) -> str:
    """Format holdings as a concise text block for the LLM prompt."""
    lines = ["CONSOLIDATED PORTFOLIO:", "-" * 50]
    for h in holdings:
        val_str = f"  (${h['value_usd']:,.0f})" if h.get("value_usd") else ""
        lines.append(
            f"  {h['ticker']:<8} | {h['investment_type']:<12} | "
            f"{h['allocation_pct']:6.2f}%{val_str}  — {h['company_name']}"
        )
    total = sum(h["allocation_pct"] for h in holdings)
    lines.append("-" * 50)
    lines.append(f"  Total: {len(holdings)} holdings, {total:.2f}% allocated")
    return "\n".join(lines)


PORTFOLIO_CONTEXT = build_portfolio_context(consolidated)


def chat_analytics(user_message: str, session_id: str = "analytics") -> str:
    """Send a message to the analytics agent with portfolio context prepended."""
    full_message = f"{PORTFOLIO_CONTEXT}\n\nUSER QUESTION:\n{user_message}"
    config   = {"configurable": {"thread_id": session_id}}
    response = analytics_graph.invoke(
        {"messages": [HumanMessage(content=full_message)]},
        config=config,
    )
    return response["messages"][-1].content


print("chat_analytics() ready")

Analytics agent graph compiled
chat_analytics() ready


### Automatic Analysis

Run the consolidation prompt immediately after loading — matches the prompt
specified in the task:

> *Can you consolidate the attached investment files and summarise using the
> format listed below ...*

In [12]:
QUERIES = [
    ("consolidated_view",
     "Please provide a consolidated view of my portfolio."),
    ("biggest_gainers",
     "Which stock or ETF have I gained the most? Please summarize your findings."),
    ("biggest_losers",
     "Which stock or ETF have I lost the most?"),
    ("overweight",
     "Which stocks are overweight or underweight?"),
]

for mode, prompt in QUERIES:
    print(DIVIDER)
    print(f"Q: {prompt}")
    print(DIVIDER)
    result = chat_analytics(prompt, session_id=f"auto_{mode}")
    print(result)
    print()



Q: Please provide a consolidated view of my portfolio.
### Consolidated Portfolio View

**Portfolio Summary:**
- Total holdings: 17
- Total Assets Under Management (AUM): $979,963
- Total gain: $135,237 (16.01% return)

---

### Holdings Overview

| Ticker | Type       | Allocation % | Value (USD) | Gain/Loss $ | Gain/Loss % |
|--------|------------|--------------|-------------|-------------|-------------|
| VTI    | ETF        | 20.85        | $204,294    | $19,912     | 11.64%      |
| NVDA   | Stock      | 12.48        | $122,321    | $64,027     | 108.7%      |
| BND    | Bond ETF   | 11.13        | $109,095    | $570        | 0.89%       |
| MSFT   | Stock      | 10.94        | $107,207    | $26,230     | 30.67%      |
| AAPL   | Stock      | 8.57         | $84,022     | $14,901     | 21.54%      |
| IJR    | ETF        | 6.2          | $60,742     | $3,545      | 6.2%        |
| VEA    | ETF        | 5.89         | $57,752     | $3,844      | 6.87%       |
| VTTHX  | Other      |

## Preset Analytics Queries

Run common portfolio questions against the consolidated holdings.

In [ ]:
DEMO_QUERIES = [
    "How well diversified is this portfolio across sectors and geographies?",
    "Are there any overlapping holdings or concentration risks I should address?",
    "How has this portfolio performed compared to the S&P 500 over the past year?",
]

for i, query in enumerate(DEMO_QUERIES, 1):
    print(DIVIDER)
    print(f"DEMO QUERY {i}: {query}")
    print(DIVIDER)
    result = chat_analytics(query, session_id=f"demo_{i}")
    print(result)
    print()

## Interactive Analytics Chat

Ask any question about your consolidated portfolio.
The agent has full context of all holdings and can search the web for
current data. Type `quit` to exit.

In [ ]:
print(DIVIDER)
print("PORTFOLIO CONSOLIDATOR & ANALYST — Interactive Mode")
print(DIVIDER)
print("Try: 'How is my portfolio doing?' / 'Should I rebalance?' / 'What is my risk exposure?'")
print("Type 'quit' to stop.")
print(DIVIDER)

while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        print("\nSession ended.")
        break
    response = chat_analytics(user_input, session_id="interactive")
    print(f"\nAgent: {response}")

## Summary

### What Changed from the Previous Version

| Feature | Before (hardcoded) | Now (dynamic) |
|---------|-------------------|---------------|
| File discovery | 5 hard-coded paths | Scans any folder automatically |
| Number of files | Fixed at 5 | Any number |
| File formats | JSON only | JSON, Excel, CSV, PDF |
| Ticker deduplication | Manual | Automatic across all files |
| Re-normalisation | Not needed | Auto re-normalises to 100% |
| User config | Edit code | Change 1 variable: `PORTFOLIO_FOLDER` |

### Architecture
```
PORTFOLIO_FOLDER  (any path, any number of files)
        │
        ▼
  discover_portfolio_files()     ← glob all supported extensions
        │
        ▼
  load_portfolio_file()          ← JSON / Excel / CSV / PDF loaders
        │
        ▼
  consolidate_holdings()         ← merge duplicates, re-normalise
        │
  ┌─────┴───────┐
  │             │
 save()     analytics agent     ← LLM + web search (tool-calling loop)
  │             │
  ▼             ▼
consolidated_   analysis
portfolio.json  report
```

### Output Contract
```json
{
  "source_folder": "../data/user1",
  "source_files": ["Portfolio1.pdf", "Portfolio2.xlsx"],
  "total_holdings": 16,
  "total_allocation": 100.0,
  "total_value_usd": 500000,
  "holdings": [
    {
      "ticker": "VTI",
      "company_name": "Vanguard Total Stock Market ETF",
      "investment_type": "ETF",
      "allocation_pct": 18.5,
      "value_usd": 92500,
      "source_files": ["Portfolio1.pdf", "Portfolio3.xlsx"]
    }
  ]
}
```

### Next Steps
- **Notebook 13**: Investment Education Agent reads `consolidated_portfolio.json`
- **Notebook 14**: Sequential multi-agent hub chains NB11 → NB12 → NB13